# Compare DQN Variants (from training_notes.txt)

Notebook này đọc `logs/<ALGO>/training_notes.txt` để tổng hợp:
- **Training Time** (tổng phút, cộng các lần train/gen)
- **Baseline Avg Time** (Random)
- **Final Avg Time** (final model được chọn)
- **Improve (%)** so với baseline: `Improve = (baseline - final) / baseline * 100`
  - Nếu avg time giảm → Improve dương (cải tiến)
  - Nếu avg time tăng → Improve âm (worse)


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
LOG_ROOT = os.path.join(ROOT, 'logs')

# Map tên thuật toán (hiển thị) -> danh sách folder có thể có trong logs/
FOLDER_CANDIDATES = {
    'DQN': ['DQN'],
    'DDQN': ['DDQN'],
    'Dueling': ['Dueling', 'DuelDQN', 'Duel_DQN', 'DuelDQNAgent', 'DuelDQN'],
    'PerDQN': ['PerDQN', 'PER_DQN', 'PERDQN'],
    'MultiStepDQN': ['MultiStepDQN', 'MultiStep_DQN', 'MultiStep'],
    'Rainbow': ['Rainbow'],
}

# Regex parser (linh hoạt theo nhiều format)
RE_BASELINE_1 = re.compile(r'Random\s*\(Baseline\)\s*:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_BASELINE_2 = re.compile(r'Baseline\s*\(Random\)\s*:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_FINAL = re.compile(r'Final Model:.*?Avg Time:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_TRAIN_MIN = re.compile(r'Training Time:\s*([0-9]+(?:\.[0-9]+)?)\s*minutes', re.IGNORECASE)

def find_algo_dir(algo_display_name: str):
    for folder in FOLDER_CANDIDATES.get(algo_display_name, []):
        p = os.path.join(LOG_ROOT, folder)
        if os.path.isdir(p):
            return p
    return None

def parse_training_notes(text: str):
    # Baseline
    baseline = None
    m = RE_BASELINE_1.search(text)
    if m:
        baseline = float(m.group(1))
    else:
        m = RE_BASELINE_2.search(text)
        if m:
            baseline = float(m.group(1))

    # Final Avg Time: lấy cái cuối cùng trong file (latest)
    finals = [float(x) for x in RE_FINAL.findall(text)]
    final_avg = finals[-1] if finals else None

    # Training time: có thể xuất hiện nhiều lần (mỗi gen), ta cộng lại
    train_mins = [float(x) for x in RE_TRAIN_MIN.findall(text)]
    total_train_minutes = float(np.sum(train_mins)) if train_mins else None

    return baseline, final_avg, total_train_minutes

rows = []
for algo in FOLDER_CANDIDATES.keys():
    algo_dir = find_algo_dir(algo)
    note_path = os.path.join(algo_dir, 'training_notes.txt') if algo_dir else None

    note_found = bool(note_path and os.path.exists(note_path))
    baseline = final_avg = total_train_minutes = None

    if note_found:
        with open(note_path, 'r', encoding='utf-8') as f:
            text = f.read()
        baseline, final_avg, total_train_minutes = parse_training_notes(text)

    improve_pct = None
    if baseline is not None and final_avg is not None and baseline != 0:
        improve_pct = (baseline - final_avg) / baseline * 100.0

    rows.append({
        'Algorithm': algo,
        'NotesFound': note_found,
        'NotesPath': note_path if note_path else '',
        'BaselineAvgTime': baseline,
        'FinalAvgTime': final_avg,
        'ImprovePct': improve_pct,
        'TrainTimeMinutes': total_train_minutes,
    })

df = pd.DataFrame(rows)
df


In [ ]:
def fmt_improve(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return 'N/A'
    if x >= 0:
        return f'Improve +{x:.2f}%'
    return f'Worse {x:.2f}%'

display_df = df.copy()
display_df['Improve(%)'] = display_df['ImprovePct'].apply(fmt_improve)
display_df = display_df.drop(columns=['ImprovePct'])
display_df


In [ ]:
plot_df = df[df['NotesFound']].copy()
plot_df = plot_df.dropna(subset=['ImprovePct'])

if len(plot_df) == 0:
    print('Chưa đủ dữ liệu để vẽ: không parse được baseline/final từ training_notes.txt.')
    print('Gợi ý: kiểm tra training_notes.txt có các dòng dạng:')
    print('  Random (Baseline): <number>')
    print('  Final Model: ... Avg Time: <number>')
else:
    plot_df = plot_df.sort_values('ImprovePct', ascending=False)

    # Improve plot
    plt.figure(figsize=(10, 4))
    plt.bar(plot_df['Algorithm'], plot_df['ImprovePct'])
    plt.axhline(0, color='black', linewidth=1)
    plt.ylabel('Improve vs Baseline (%)')
    plt.title('So sánh hiệu năng (Avg Time) so với Baseline')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    # Training time plot (if available)
    tt = df[df['NotesFound']].dropna(subset=['TrainTimeMinutes']).copy()
    if len(tt) > 0:
        tt = tt.sort_values('TrainTimeMinutes', ascending=False)
        plt.figure(figsize=(10, 4))
        plt.bar(tt['Algorithm'], tt['TrainTimeMinutes'])
        plt.ylabel('Training Time (minutes)')
        plt.title('So sánh thời gian train (tổng phút)')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
